# MSO Foundations — engineer's lab

Short experiments for the authenticated module. Start with the [module index](../course%20content%20HTML/01-mso-foundations/index.html#module-index) and [authenticated lesson](https://anthropic-partners.skilljar.com/path/claude-certified-developer-foundations/mso-foundations/486742/scorm/1zuxexjatih0p).

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'study_support.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from study_support import load_anthropic_api_key, messages_create
assert callable(load_anthropic_api_key)
assert callable(messages_create)

## 1. Tokens and context are a budget

Input context and generated output compete for a fixed window. The next cell uses a deliberately rough character proxy so it stays offline; real applications should use authoritative counts. [Tokens source](../course%20content%20HTML/01-mso-foundations/02-how-llms-behave.html#tokens-the-unit-of-input-output-and-cost) · [Context-window source](../course%20content%20HTML/01-mso-foundations/02-how-llms-behave.html#the-context-window-a-fixed-budget)

In [ ]:
CONTEXT_BUDGET = 200
PROMPT = "Classify this support ticket and explain the decision briefly."
RESERVED_OUTPUT = 60

def rough_token_proxy(text):
    return (len(text) + 3) // 4

input_tokens = rough_token_proxy(PROMPT)
remaining = CONTEXT_BUDGET - input_tokens - RESERVED_OUTPUT
print({"input_proxy": input_tokens, "reserved_output": RESERVED_OUTPUT, "headroom": remaining})
assert input_tokens + RESERVED_OUTPUT + remaining == CONTEXT_BUDGET
assert remaining >= 0, "Prompt plus reserved output exceeds the context budget"

## 2. Variation changes testing; examples change cost

Sampling permits multiple plausible outputs, so robust evals check properties rather than one exact sentence. Zero-, one-, and multi-shot prompts trade more context and input cost for clearer demonstrations. [Non-determinism source](../course%20content%20HTML/01-mso-foundations/02-how-llms-behave.html#non-determinism-what-it-means-for-testing-and-evals) · [Prompting source](../course%20content%20HTML/01-mso-foundations/04-prompting-modes.html#the-three-modes)

In [ ]:
import random

choices = ["refund", "replace", "escalate"]
samples = [random.Random(seed).choice(choices) for seed in range(8)]
print("simulated sampled outputs:", samples)
assert len(set(samples)) > 1

TASK = "Classify the ticket."
EXAMPLES = ["Broken on arrival -> replace", "Charged twice -> refund"]
prompts = {
    "zero-shot": TASK,
    "one-shot": TASK + "\n" + EXAMPLES[0],
    "multi-shot": TASK + "\n" + "\n".join(EXAMPLES),
}
cost_proxy = {name: rough_token_proxy(text) for name, text in prompts.items()}
print("prompt token proxies:", cost_proxy)
assert cost_proxy["zero-shot"] < cost_proxy["one-shot"] < cost_proxy["multi-shot"]

## 3. Choose independent knobs, then an access pattern

A model tier sets a capability/latency/cost baseline; reasoning mode controls extra inference effort for that model. Separately, choose synchronous calls for simplicity, streaming for incremental user-visible output, async concurrency for independent requests, or batch for latency-tolerant volume. [Model and reasoning source](../course%20content%20HTML/01-mso-foundations/03-models-reasoning.html#reasoning-modes-are-a-separate-setting-from-model-choice) · [Access-pattern source](../course%20content%20HTML/01-mso-foundations/05-technical-substrate.html#asynchronous-patterns-for-high-volume-work)

In [ ]:
def access_pattern(*, incremental=False, high_volume=False, latency_tolerant=False):
    if high_volume and latency_tolerant:
        return "batch"
    if incremental:
        return "streaming"
    if high_volume:
        return "async"
    return "synchronous"

MODEL_TIER = "balanced"
REASONING_MODE = "standard"
assert MODEL_TIER != REASONING_MODE
assert access_pattern(incremental=True) == "streaming"
assert access_pattern(high_volume=True) == "async"
assert access_pattern(high_volume=True, latency_tolerant=True) == "batch"
assert access_pattern() == "synchronous"

In [ ]:
RUN_LIVE = False
LIVE_MODEL = ""

if RUN_LIVE:
    assert LIVE_MODEL, "Set LIVE_MODEL to a current model ID"
    response = messages_create({
        "model": LIVE_MODEL,
        "max_tokens": 64,
        "messages": [{"role": "user", "content": "Give one reason to test properties instead of exact wording."}],
    })
    print(response["content"][0]["text"])
else:
    print("Live call skipped; offline exercises are complete.")

## Hands-on exercise

Using [model choice and reasoning mode](../course%20content%20HTML/01-mso-foundations/03-models-reasoning.html#how-the-two-work-together) and [access patterns](../course%20content%20HTML/01-mso-foundations/05-technical-substrate.html#synchronous-streaming-and-real-time-responses), add one workload to the previous cell. State its latency, cost, and quality constraints; choose the model tier, reasoning mode, and access pattern; then add an assertion that captures your decision. Compare your choice with the [module exercise](../course%20content%20HTML/01-mso-foundations/06-module-wrap-up.html#screen-7--exercise-predict-the-behavior).